In [3]:
import torch 
import torch.nn as nn

# in_features: 입력 특성(또는 차원)의 수입니다.
# out_features: 출력 특성(또는 차원)의 수입니다.
# bias: 편향을 사용할지 여부를 결정하는 부울 값입니다. 기본값은 True이며, 이 경우 학습 가능한 편향이 추가됩니다.
linear_layer = nn.Linear(in_features=10, out_features=5, bias=True)

# 임의의 데이터 생성
# 예시를 위해 배치 크기가 1, 입력 특성의 개수가 10인 텐서 생성
input_tensor = torch.randn(1, 10)

# 생성한 Linear 레이어에 입력 데이터를 전달하여 출력 데이터를 얻음
output_tensor = linear_layer(input_tensor)

print('입력 층:', input_tensor.size())
print('출력 층:', output_tensor.size())
print('출력 결과: ', output_tensor)

입력 층: torch.Size([1, 10])
출력 층: torch.Size([1, 5])
출력 결과:  tensor([[ 0.6168,  1.0085,  0.4037,  0.3051, -0.3405]],
       grad_fn=<AddmmBackward0>)


In [4]:
# 합성 곱 층 (Convolutional Layer) 생성
# in_channels: 입력 채널의 수입니다. 
# out_channels: 출력 채널의 수입니다.
# kernel_size: 커널(필터)의 크기입니다. 정수 혹은 (높이, 너비) 형태의 튜플을 사용할 수 있습니다.
# stride: 필터를 적용하는 스트라이드의 크기입니다. 기본값은 1입니다.
# padding: 입력의 각 측면에 추가할 패딩의 크기입니다. 기본값은 0입니다.
# bias: 편향을 사용할지 여부를 결정하는 부울 값입니다. 기본값은 True입니다.

conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=0, bias=True)

# 합성곱층(Convolutional Layer)
-  이미지 인식 인공지능(CNN)의 핵심이자 **'AI의 눈'** 역할을 하는 층입니다. 직관적인 비유와 파이썬 코드로 아주 쉽게 풀어드릴게요.

---

### 🔍 1. 직관적 이해: "돋보기로 단서 찾기"

거대한 풍경 사진에서 '고양이'를 찾는다고 상상해 보세요. 우리는 사진 전체를 한 번에 통째로 분석하지 않습니다. 시선을 좁혀서 뾰족한 귀 모양, 수염, 꼬리 같은 '특징(패턴)'을 부분부분 훑어보며 찾아냅니다.

합성곱층이 바로 이 방식을 똑같이 사용합니다!

* **입력 이미지 (Input):** 숫자로 이루어진 거대한 픽셀 표 (예: 100x100 크기)
* **필터 / 커널 (Filter / Kernel):** AI가 들고 있는 '특수 돋보기'입니다. (보통 3x3 크기) 어떤 돋보기는 '가로선'만 찾고, 어떤 돋보기는 '세로선'만 찾습니다. 이것이 바로 우리가 학습시켜야 할 **가중치($w$)** 뭉치입니다.
* **스트라이드 (Stride):** 돋보기를 몇 칸씩 이동시키며 사진을 훑을지 결정하는 보폭입니다.
* **합성곱 연산:** 돋보기를 사진 위에 올려놓고, **겹치는 부분의 픽셀 숫자와 돋보기의 가중치 숫자를 각각 곱해서 싹 다 더합니다.** (네, 맞습니다! 우리가 앞에서 배웠던 '선형 결합(내적)'을 사진 전체를 훑으며 쪼개서 하는 겁니다.)
* **특성 맵 (Feature Map):** 사진 전체를 돋보기로 다 훑고 나서 만들어진 새로운 압축 이미지입니다. "아! 이 부분에 가로선 패턴이 강하게 있네!"라는 정보가 담긴 결과물입니다.

---

### 💻 2. 파이썬 예제 코드로 보는 합성곱의 원리

실제 딥러닝 프레임워크(PyTorch 등)에서는 이 과정을 함수 한 줄(`nn.Conv2d`)로 끝내지만, 속에서 어떤 수학적 연산이 일어나는지 직관적으로 볼 수 있도록 `numpy` 라이브러리만 사용해서 밑바닥부터 구현해 보았습니다.

```python
import numpy as np

# 1. 입력 이미지 생성 (5x5 크기)
# (예: 1은 선이 있는 부분, 0은 빈 공간이라고 상상해 보세요)
image = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0]
])

# 2. 필터(커널) 생성 (3x3 크기)
# 이 필터는 대각선(\) 패턴을 찾는 데 특화된 가중치를 가졌다고 가정합니다.
filter_kernel = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])

# 3. 특성 맵(결과물)을 담을 빈 도화지 준비
# 5x5 이미지에 3x3 돋보기를 1칸씩 움직이면, 3x3 크기의 결과물이 나옵니다.
feature_map = np.zeros((3, 3))

print("--- 돋보기(필터) 훑기 시작! ---\n")

# 4. 합성곱 연산 (컨베이어 벨트처럼 훑기)
for row in range(3):       # 세로로 이동 (Stride 1)
    for col in range(3):   # 가로로 이동 (Stride 1)
        
        # 돋보기가 올라간 부분의 이미지 잘라내기 (3x3 픽셀)
        window = image[row : row+3, col : col+3]
        
        # 합성곱 계산: 잘라낸 이미지와 필터의 같은 위치 숫자끼리 곱하고(Element-wise), 싹 다 더하기
        convolution_sum = np.sum(window * filter_kernel)
        
        # 결과를 특성 맵에 기록
        feature_map[row, col] = convolution_sum
        
        print(f"위치(행:{row}, 열:{col}) 계산 결과: {convolution_sum}")

print("\n--- 최종 완성된 특성 맵 (Feature Map) ---")
print(feature_map)

```

---

### 🧠 3. 기존의 '은닉층(Dense Layer)'과 무엇이 다를까?

이전에 배우셨던 일반적인 신경망(완전연결층)은 이미지를 1줄로 길게 쫙 펴서(1차원 배열) 입력으로 받습니다.

* **기존 방식의 문제점:** 이미지를 1줄로 펴버리면, "눈 바로 밑에 코가 있다"는 **'공간적 정보(2차원 구조)'가 완전히 박살 납니다.** 또한 픽셀이 100만 개라면 가중치도 100만 개가 필요해서 컴퓨터가 터져버립니다.
* **합성곱층의 해결책:** 1. 2차원 모양을 그대로 유지한 채 필터를 갖다 댑니다. (공간 정보 보존)
2. 하나의 돋보기(예: 3x3 가중치 9개)를 이리저리 옮기며 재사용합니다. 즉, 가중치 단 9개만으로 이미지 전체를 훑을 수 있어 **파라미터(학습할 가중치)의 수를 압도적으로 줄여줍니다.**

결과적으로 합성곱층은 "이전 질문에서 배우셨던 벡터의 내적($w \cdot x$)을 한 번에 크게 하는 게 아니라, 작은 돋보기 크기로 쪼개서 사진 전체를 스캔하듯 반복하는 과정"일 뿐입니다!

합성곱층(Convolution Layer)은 이미지에서 **작은 필터(커널)** 를 슬라이딩하면서,  
엣지/텍스처 같은 지역 패턴을 추출하는 층입니다.  
완전연결층보다 파라미터 수가 적고, 공간 구조(가까운 픽셀 관계)를 잘 보존합니다.

예를 들어 아래 코드가 있다고 보면:

```python
import torch
import torch.nn as nn

x = torch.randn(8, 3, 32, 32)  # (배치, 채널, 높이, 너비)

conv = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1,
    dilation=1,
    groups=1,
    bias=True
)

y = conv(x)
print(y.shape)  # torch.Size([8, 16, 32, 32])
```

각 파라미터 의미:
1. `in_channels=3`  
입력 채널 수입니다. RGB 이미지면 보통 3입니다.

2. `out_channels=16`  
출력 특성맵 개수(필터 개수)입니다.  
즉 “16개의 다른 패턴 탐지기”를 둔다는 뜻입니다.

3. `kernel_size=3`  
필터 크기입니다. \(3\times3\) 영역을 보며 특징을 뽑습니다.

4. `stride=1`  
필터 이동 간격입니다. 1이면 한 칸씩 촘촘히 이동, 2면 절반 정도로 다운샘플 효과가 납니다.

5. `padding=1`  
입력 가장자리에 0을 둘러 크기 손실을 줄입니다.  
\(3\times3\), stride=1일 때 padding=1이면 보통 공간 크기(H,W) 유지됩니다.

6. `dilation=1`  
커널 내부 간격입니다. 1은 기본, 2 이상이면 더 넓은 수용영역을 봅니다.

7. `groups=1`  
채널 연결 방식입니다.  
기본 1은 모든 입력 채널을 함께 사용,  
`groups=in_channels`이면 depthwise convolution이 됩니다.

8. `bias=True`  
출력 채널마다 편향 \(b\)를 추가할지 여부입니다.

출력 크기 공식(2D):

$$
H_{out}=\left\lfloor\frac{H+2P-D(K-1)-1}{S}+1\right\rfloor,\quad
W_{out}=\left\lfloor\frac{W+2P-D(K-1)-1}{S}+1\right\rfloor
$$

- \(K\): kernel size, \(S\): stride, \(P\): padding, \(D\): dilation

지금 예제 값에서는 \(K=3, S=1, P=1, D=1\) 이라서 입력 \(32\times32\)가 그대로 \(32\times32\)로 유지됩니다.  
채널만 `3 -> 16`으로 바뀝니다.

# Pooling Layer ( 풀링 층 )

- 특정 맵의 크기를 줄이거나 요약, 계산량을 감소시키고, 과적합을 방지 
- 최대 풀링과 평균 풀링이 자주 사용됨. 

In [6]:
# kernel_size: 풀링을 적용할 윈도우의 크기입니다.
# stride: 윈도우를 이동시키는 스트라이드의 크기입니다. 기본값은 kernel_size와 같습니다.
# padding: 입력의 각 측면에 추가할 패딩의 크기입니다. 기본값은 0입니다.

max_pool = nn.MaxPool2d(kernel_size=2, stride=1, padding=0)
# 임의의 이미지 데이터 생성
# 예시를 위해 배치 크기가 1, 채널 수가 16, 이미지 크기가 32x32인 텐서 생성
input_tensor = torch.randn(1, 16, 32, 32)

# 생성한 MaxPool2d 레이어에 입력 데이터를 전달하여 출력 데이터를 얻음
output_tensor = max_pool(input_tensor)

print('입력 층:', input_tensor.size())
print('출력 층:', output_tensor.size())
#print('출력 결과: ', output_tensor)

입력 층: torch.Size([1, 16, 32, 32])
출력 층: torch.Size([1, 16, 31, 31])


# 순환 층 (Recurrent Layers) 

- 시간에 따른 정보의 흐름을 모델링하는데 사용
- 과거의 정보를 기억하며, 시계열 데이터나 자연어 처리에 주로 사용 됨. 
- 순환층을 사용하는 모델로는 `RNN`, `LSTM`, `GRU` 등이 있음. 

In [8]:
# RNN 레이어 생성
# input_size: 입력 특성의 개수
# hidden_size: 숨겨진 상태의 특성 개수
# num_layers: RNN 층의 수
# bias: 바이어스 사용 여부
# batch_first: 입력 및 출력 텐서의 첫 번째 차원이 배치 크기인지 여부

rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, bias=True, batch_first=False) 

# 예시를 위해 시퀀스 길이가 5, 배치 크기가 3, 입력 크기가 10인 텐서 생성
input_seq = torch.randn(5, 3, 10)

# 상태의 크기는 (num_layers, batch, hidden_size)
hidden_state = torch.randn(2, 3, 20)

# RNN에 입력 시퀀스를 전달하고, 출력과 최종 숨겨진 상태를 얻음
output_seq, final_hidden_state = rnn(input_seq, hidden_state)

print('입력 층:', input_seq.size())
print('히든 층:', hidden_state.size())
print('최종 히든 층:', final_hidden_state.size())
print('출력 층:', output_seq.size())
# print('출력 결과: ', output_seq)

입력 층: torch.Size([5, 3, 10])
히든 층: torch.Size([2, 3, 20])
최종 히든 층: torch.Size([2, 3, 20])
출력 층: torch.Size([5, 3, 20])
